# Лабораторная работа № 6 - Генерация текста с помощью RNN

В данной лабораторной работе мы реализуем рекуррентную нейронную сеть (RNN) для генерации текста на основе художественного произведения.

## Задачи:
1. Скачать художественное произведение
2. Создать датасет на основе произведения
3. Создать и обучить модель RNN для генерации текста
4. Написать функцию генерации текста

In [1]:
import nltk
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import requests
from collections import Counter
import pickle
import os

nltk.download('punkt', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используемое устройство: {device}')

Используемое устройство: cpu


[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1002)>


In [4]:
# Установка корректного certifi для HTTPS-запросов
%pip install --force-reinstall certifi

import certifi
import os

# Установка переменной окружения для requests
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# Загрузка художественного произведения - "Война и мир" Л.Н. Толстого
response = requests.get(url)
text = response.text

# Очистка текста
start_marker = "CHAPTER I"
end_marker = "End of the Project Gutenberg"

start_idx = text.find(start_marker)
end_idx = text.find(end_marker)

if start_idx != -1 and end_idx != -1:
    text = text[start_idx:end_idx]
else:
    # Если маркеры не найдены, берем первые 500000 символов
    text = text[:500000]

print(f"Длина текста: {len(text)} символов")
print(f"Первые 500 символов:\n{text[:500]}")

ERROR: Could not install packages due to an OSError: Could not find a suitable TLS CA certificate bundle, invalid path: /Users/Gret/Desktop/DataManager/scientific-api/.venv/lib/python3.11/site-packages/certifi/cacert.pem


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Длина текста: 500000 символов
Первые 500 символов:
﻿The Project Gutenberg eBook of War and Peace, by Leo Tolstoy

This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online at
www.gutenberg.org. If you are not located in the United States, you
will have to check the laws of the country where you are located before
usin


In [5]:
# Предобработка текста
def preprocess_text(text):
    # Удаляем лишние символы и приводим к нижнему регистру
    text = re.sub(r'[^a-zA-Z\s.,!?;:]', '', text)
    text = text.lower()
    # Заменяем множественные пробелы на одинарные
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

clean_text = preprocess_text(text)
print(f"Длина очищенного текста: {len(clean_text)} символов")

# Создание словаря символов
chars = sorted(list(set(clean_text)))
vocab_size = len(chars)
print(f"Размер словаря: {vocab_size}")
print(f"Символы: {chars}")

# Создание маппингов символ -> индекс и индекс -> символ
char_to_idx = {char: idx for idx, char in enumerate(chars)}
idx_to_char = {idx: char for idx, char in enumerate(chars)}

print(f"Пример маппинга: 'a' -> {char_to_idx['a']}, {char_to_idx['a']} -> '{idx_to_char[char_to_idx['a']]}'")

Длина очищенного текста: 476563 символов
Размер словаря: 33
Символы: [' ', '!', ',', '.', ':', ';', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Пример маппинга: 'a' -> 7, 7 -> 'a'


In [6]:
# Параметры для создания датасета
seq_length = 50  # Длина последовательности
batch_size = 64

class TextDataset(Dataset):
    def __init__(self, text, char_to_idx, seq_length):
        self.text = text
        self.char_to_idx = char_to_idx
        self.seq_length = seq_length
        
        # Преобразуем текст в индексы
        self.encoded_text = [char_to_idx[char] for char in text]
        
    def __len__(self):
        return len(self.encoded_text) - self.seq_length
    
    def __getitem__(self, idx):
        # Входная последовательность
        input_seq = self.encoded_text[idx:idx + self.seq_length]
        # Целевая последовательность (сдвинутая на 1)
        target_seq = self.encoded_text[idx + 1:idx + self.seq_length + 1]
        
        return torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long)

# Создание датасета и загрузчика данных
dataset = TextDataset(clean_text, char_to_idx, seq_length)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Размер датасета: {len(dataset)}")
print(f"Количество батчей: {len(data_loader)}")

# Пример данных
input_sample, target_sample = dataset[0]
print(f"\nПример входной последовательности: {input_sample[:10]}")
print(f"Соответствующий текст: '{''.join([idx_to_char[idx.item()] for idx in input_sample[:10]])}'")
print(f"Пример целевой последовательности: {target_sample[:10]}")
print(f"Соответствующий текст: '{''.join([idx_to_char[idx.item()] for idx in target_sample[:10]])}'")

Размер датасета: 476513
Количество батчей: 7446

Пример входной последовательности: tensor([26, 14, 11,  0, 22, 24, 21, 16, 11,  9])
Соответствующий текст: 'the projec'
Пример целевой последовательности: tensor([14, 11,  0, 22, 24, 21, 16, 11,  9, 26])
Соответствующий текст: 'he project'


In [7]:
class RNNTextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, dropout=0.5):
        super(RNNTextGenerator, self).__init__()
        
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Слой эмбеддингов
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # LSTM слои
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, 
                           batch_first=True, dropout=dropout)
        
        # Dropout для регуляризации
        self.dropout = nn.Dropout(dropout)
        
        # Выходной слой
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden=None):
        # x: (batch_size, seq_length)
        
        # Эмбеддинги
        embedded = self.embedding(x)  # (batch_size, seq_length, embed_size)
        
        # LSTM
        lstm_out, hidden = self.lstm(embedded, hidden)  # (batch_size, seq_length, hidden_size)
        
        # Dropout
        lstm_out = self.dropout(lstm_out)
        
        # Преобразование для выходного слоя
        # Reshape: (batch_size * seq_length, hidden_size)
        lstm_out = lstm_out.contiguous().view(-1, self.hidden_size)
        
        # Выходной слой
        output = self.fc(lstm_out)  # (batch_size * seq_length, vocab_size)
        
        return output, hidden
    
    def init_hidden(self, batch_size):
        """Инициализация скрытого состояния"""
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        return (h0, c0)

# Параметры модели
embed_size = 128
hidden_size = 256
num_layers = 2
dropout = 0.5
learning_rate = 0.001
num_epochs = 20

# Создание модели
model = RNNTextGenerator(vocab_size, embed_size, hidden_size, num_layers, dropout).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Модель создана. Параметры:")
print(f"Vocab size: {vocab_size}")
print(f"Embed size: {embed_size}")
print(f"Hidden size: {hidden_size}")
print(f"Num layers: {num_layers}")
print(f"Dropout: {dropout}")
print(f"\nОбщее количество параметров: {sum(p.numel() for p in model.parameters())}")

Модель создана. Параметры:
Vocab size: 33
Embed size: 128
Hidden size: 256
Num layers: 2
Dropout: 0.5

Общее количество параметров: 934305


In [8]:
def train_model(model, data_loader, criterion, optimizer, num_epochs, device):
    model.train()
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        hidden = model.init_hidden(batch_size)
        
        for batch_idx, (input_seq, target_seq) in enumerate(data_loader):
            # Если размер батча меньше заданного, пропускаем
            if input_seq.size(0) != batch_size:
                continue
                
            input_seq = input_seq.to(device)
            target_seq = target_seq.to(device)
            
            # Обнуление градиентов
            optimizer.zero_grad()
            
            # Отсоединение скрытого состояния от графа вычислений
            hidden = tuple([h.detach() for h in hidden])
            
            # Прямой проход
            output, hidden = model(input_seq, hidden)
            
            # Вычисление потерь
            # target_seq нужно "растянуть" для соответствия output
            target_flat = target_seq.contiguous().view(-1)
            loss = criterion(output, target_flat)
            
            # Обратное распространение
            loss.backward()
            
            # Обрезка градиентов для предотвращения взрыва градиентов
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
            
            # Обновление весов
            optimizer.step()
            
            epoch_loss += loss.item()
            
            # Вывод статистики каждые 100 батчей
            if batch_idx % 100 == 0:
                print(f'Эпоха [{epoch+1}/{num_epochs}], Батч [{batch_idx}/{len(data_loader)}], '
                      f'Потери: {loss.item():.4f}')
        
        avg_loss = epoch_loss / len(data_loader)
        losses.append(avg_loss)
        print(f'Эпоха [{epoch+1}/{num_epochs}] завершена. Средние потери: {avg_loss:.4f}')
        
        # Сохранение промежуточной модели каждые 5 эпох
        if (epoch + 1) % 5 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
                'char_to_idx': char_to_idx,
                'idx_to_char': idx_to_char,
                'vocab_size': vocab_size
            }, f'rnn_checkpoint_epoch_{epoch+1}.pth')
            print(f'Модель сохранена: rnn_checkpoint_epoch_{epoch+1}.pth')
    
    return losses

print("Функция обучения создана. Готов к началу обучения.")

Функция обучения создана. Готов к началу обучения.


In [9]:
# Обучение модели
print("Начинаем обучение модели...")
losses = train_model(model, data_loader, criterion, optimizer, num_epochs, device)

# Визуализация потерь
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(losses) + 1), losses, 'b-', linewidth=2)
plt.title('Функция потерь во время обучения RNN')
plt.xlabel('Эпоха')
plt.ylabel('Потери')
plt.grid(True)
plt.show()

# Сохранение финальной модели
torch.save({
    'model_state_dict': model.state_dict(),
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size,
    'embed_size': embed_size,
    'hidden_size': hidden_size,
    'num_layers': num_layers
}, 'rnn_text_generator_final.pth')

print("Обучение завершено. Финальная модель сохранена как 'rnn_text_generator_final.pth'")

Начинаем обучение модели...
Эпоха [1/20], Батч [0/7446], Потери: 3.5134
Эпоха [1/20], Батч [100/7446], Потери: 2.2029
Эпоха [1/20], Батч [200/7446], Потери: 1.9808
Эпоха [1/20], Батч [300/7446], Потери: 1.8565
Эпоха [1/20], Батч [400/7446], Потери: 1.7731
Эпоха [1/20], Батч [500/7446], Потери: 1.7191
Эпоха [1/20], Батч [600/7446], Потери: 1.7106
Эпоха [1/20], Батч [700/7446], Потери: 1.6414
Эпоха [1/20], Батч [800/7446], Потери: 1.6555
Эпоха [1/20], Батч [900/7446], Потери: 1.6363
Эпоха [1/20], Батч [1000/7446], Потери: 1.6296
Эпоха [1/20], Батч [1100/7446], Потери: 1.5790
Эпоха [1/20], Батч [1200/7446], Потери: 1.5368
Эпоха [1/20], Батч [1300/7446], Потери: 1.5656
Эпоха [1/20], Батч [1400/7446], Потери: 1.5307
Эпоха [1/20], Батч [1500/7446], Потери: 1.5558
Эпоха [1/20], Батч [1600/7446], Потери: 1.5368
Эпоха [1/20], Батч [1700/7446], Потери: 1.4814
Эпоха [1/20], Батч [1800/7446], Потери: 1.5029
Эпоха [1/20], Батч [1900/7446], Потери: 1.4897
Эпоха [1/20], Батч [2000/7446], Потери: 1.46

KeyboardInterrupt: 

In [ ]:
def generate_text(model, char_to_idx, idx_to_char, prime_str='the', sample_len=100, temperature=0.8):
    """Генерация текста с помощью обученной модели"""
    model.eval()
    
    # Преобразование начальной строки в индексы
    prime_input = [char_to_idx.get(char, 0) for char in prime_str.lower()]
    predicted = prime_str.lower()
    
    # Инициализация скрытого состояния
    hidden = model.init_hidden(1)
    
    with torch.no_grad():
        # Прогон начальной строки через модель
        for char_idx in prime_input[:-1]:
            input_tensor = torch.tensor([[char_idx]], dtype=torch.long).to(device)
            output, hidden = model(input_tensor, hidden)
        
        # Последний символ начальной строки
        input_tensor = torch.tensor([[prime_input[-1]]], dtype=torch.long).to(device)
        
        # Генерация новых символов
        for _ in range(sample_len):
            output, hidden = model(input_tensor, hidden)
            
            # Применение температуры для разнообразия
            output = output.squeeze().div(temperature).exp()
            
            # Сэмплирование следующего символа
            char_weights = output / output.sum()
            char_idx = torch.multinomial(char_weights, 1)[0].item()
            
            # Добавление предсказанного символа
            predicted += idx_to_char[char_idx]
            
            # Подготовка для следующей итерации
            input_tensor = torch.tensor([[char_idx]], dtype=torch.long).to(device)
    
    return predicted

# Функция для генерации текста с различными параметрами
def generate_samples(model, char_to_idx, idx_to_char):
    """Генерация нескольких примеров текста"""
    prime_strings = ['the', 'war', 'peace', 'love', 'time']
    temperatures = [0.5, 0.8, 1.0]
    
    for prime in prime_strings:
        print(f"\n{'='*50}")
        print(f"Начальная строка: '{prime}'")
        print(f"{'='*50}")
        
        for temp in temperatures:
            print(f"\nТемпература: {temp}")
            print("-" * 30)
            generated = generate_text(model, char_to_idx, idx_to_char, 
                                    prime_str=prime, sample_len=200, temperature=temp)
            print(generated)
            print()

print("Функции генерации текста созданы.")

In [ ]:
# Генерация примеров текста
print("Генерируем примеры текста с обученной моделью...")
generate_samples(model, char_to_idx, idx_to_char)

# Интерактивная функция генерации
def interactive_generation():
    """Интерактивная генерация текста"""
    print("\nИнтерактивная генерация текста")
    print("Введите начальную строку (или 'quit' для выхода):")
    
    while True:
        prime = input("Начальная строка: ").strip()
        if prime.lower() == 'quit':
            break
        
        try:
            length = int(input("Длина генерируемого текста (по умолчанию 150): ") or 150)
            temp = float(input("Температура (0.1-2.0, по умолчанию 0.8): ") or 0.8)
            
            generated = generate_text(model, char_to_idx, idx_to_char, 
                                    prime_str=prime, sample_len=length, temperature=temp)
            
            print("\nСгенерированный текст:")
            print("-" * 50)
            print(generated)
            print("-" * 50)
            
        except ValueError:
            print("Ошибка ввода. Попробуйте снова.")
        except Exception as e:
            print(f"Произошла ошибка: {e}")

print("Интерактивная функция создана. Можете вызвать interactive_generation() для интерактивной генерации.")

In [ ]:
# Сохранение всех результатов
results = {
    'model_info': {
        'vocab_size': vocab_size,
        'embed_size': embed_size,
        'hidden_size': hidden_size,
        'num_layers': num_layers,
        'seq_length': seq_length,
        'num_epochs': num_epochs
    },
    'training_losses': losses,
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'sample_text': clean_text[:1000]
}

with open('rnn_training_results.pkl', 'wb') as f:
    pickle.dump(results, f)